# 03 — Explainability, Tasks A / B / C

Run this **after** `02_training.ipynb`. Needs the winning checkpoints for
each task backed up to `EDOS_DATA/outputs_backup` in Drive.

LIME is the primary explainability method (proposal Section IV.E), applied
per-example across Tasks A, B, and C. SHAP is applied comparatively on the
same samples, to validate LIME's attributions; for Task C it also powers
the gender-bias audit.

Only the **best** checkpoint per (task, model) is restored from Drive, not
the whole backup: `outputs_backup/` also contains every LR/batch-size
sweep run (up to 8 configs x 2 models x 3 tasks) and Task B's
backtranslation-augmented variants. "Best" here means the final run
trained with the winning hyperparameters from the sweep, no augmentation
-- `select_best_checkpoints` in `src/explain.py` picks these out via each
checkpoint's `results.json` (`run_name == "default"`, `augment == "none"`),
so augmented variants are intentionally excluded even though they're in
the backup.

## Setup: clone repo, restore only the best checkpoints from Drive

In [ ]:
# Fresh session only — skip if continuing directly from 02_training.ipynb
from google.colab import drive
drive.mount('/content/drive')
!git clone https://github.com/Mahekd/interpretable-nlp-sexism-detection.git
%cd interpretable-nlp-sexism-detection
!pip install -q -r requirements.txt
!mkdir -p outputs


In [ ]:
import sys
sys.path.insert(0, '.')

from src.explain import copy_best_checkpoints

DRIVE_BACKUP_DIR = "/content/drive/MyDrive/EDOS_DATA/outputs_backup"

# Copies just the 6 final checkpoints (Task A/B/C x roberta-base/bert-base-uncased)
# into outputs/, skipping every sweep run and Task B's augmented variants.
selected_checkpoints = copy_best_checkpoints(DRIVE_BACKUP_DIR, output_dir="outputs")
selected_checkpoints


If any `(task, model)` pair is missing above, it means that final run
hasn't been backed up to Drive yet (or was trained with a `--run_name`,
e.g. still a sweep run) — go back to `02_training.ipynb` and run/back up
that task's final config before re-running the cell above.

In [ ]:
import os

import pandas as pd

from src.data import TASK_LABELS, build_task_frame, get_splits, load_raw
from src.explain import (
    ModelWrapper,
    bias_audit,
    build_shap_explainer,
    evaluate_faithfulness,
    explain_lime,
    explain_shap,
)

df = load_raw('data/edos_labelled_aggregated.csv')

MODEL_LABELS = {"roberta-base": "RoBERTa-base", "bert-base-uncased": "BERT-base"}


## Helper: run LIME → faithfulness → SHAP for one (task, model)

Same pattern for every task: LIME first (primary method) on a handful of
test examples, then the ERASER-style faithfulness metrics on those same
examples, then SHAP as the comparative check. Task C additionally runs the
gender-bias audit on top of the SHAP values.

In [ ]:
def run_task_explainability(task, model_name, n_samples=5, k=5, run_bias_audit=False, seed=42):
    key = (task, model_name)
    if key not in selected_checkpoints:
        print(f"Skipping task {task} / {model_name}: no best checkpoint found (see setup cell above).")
        return None

    checkpoint_dir = os.path.join("outputs", selected_checkpoints[key]["dir"])
    dev_f1 = selected_checkpoints[key]["dev_macro_f1"]
    labels = TASK_LABELS[task]
    arch_label = MODEL_LABELS.get(model_name, model_name)

    print(f"\n{'=' * 70}")
    print(f"Task {task} — {arch_label}  ({checkpoint_dir}, dev macro-F1={dev_f1:.4f})")
    print(f"{'=' * 70}")

    wrapper = ModelWrapper(checkpoint_dir)
    task_df = build_task_frame(df, task)
    _, _, test_df = get_splits(task_df)
    sample = test_df.sample(n=min(n_samples, len(test_df)), random_state=seed)
    sample_texts = sample['text'].tolist()

    print("\n--- LIME (primary) ---")
    for text in sample_texts:
        exp, pred = explain_lime(text, wrapper, labels)
        print(f"\n{text}\n  -> predicted: {labels[pred]}")
        exp.show_in_notebook(text=True, labels=(pred,))

    print(f"\n--- Faithfulness (comprehensiveness / sufficiency), k={k} ---")
    faithfulness = evaluate_faithfulness(sample_texts, wrapper, labels, k=k)
    faithfulness_df = pd.DataFrame(faithfulness)
    display(faithfulness_df)
    print(f"Mean comprehensiveness: {faithfulness_df['comprehensiveness'].mean():.4f} (higher = more faithful)")
    print(f"Mean sufficiency:       {faithfulness_df['sufficiency'].mean():.4f} (lower = more faithful)")

    print("\n--- SHAP (comparative validation) ---")
    shap_explainer = build_shap_explainer(wrapper, labels)
    shap_values = explain_shap(sample_texts, shap_explainer)
    import shap
    shap.plots.text(shap_values[0])

    bias_rows = None
    if run_bias_audit:
        print("\n--- Bias audit (gender-neutral term SHAP attribution) ---")
        bias_rows = bias_audit(sample_texts, shap_values)
        display(pd.DataFrame(bias_rows).head(10))

    return {
        "checkpoint_dir": checkpoint_dir,
        "faithfulness": faithfulness_df,
        "shap_values": shap_values,
        "bias_audit": bias_rows,
    }


explainability_results = {}


## Task A — binary sexism classification

LIME explanations, faithfulness, and comparative SHAP, for both
architectures' best checkpoint.

In [ ]:
for model_name in ["roberta-base", "bert-base-uncased"]:
    explainability_results[("A", model_name)] = run_task_explainability("A", model_name)


## Task B — category classification (4-way)

Same pattern as Task A, on the winning (non-augmented) checkpoint per
architecture.

In [ ]:
for model_name in ["roberta-base", "bert-base-uncased"]:
    explainability_results[("B", model_name)] = run_task_explainability("B", model_name)


## Task C — vector classification (11-way) + gender-bias audit

Same LIME → faithfulness → SHAP pattern, plus the SHAP-based bias audit
(Muntasir & Noor, 2024) checking whether gender-neutral terms (e.g.
"woman", "he", "wife") get disproportionate attribution on their own —
a sign of a spurious gender-to-sexism correlation rather than genuine
sexist content.

In [ ]:
for model_name in ["roberta-base", "bert-base-uncased"]:
    explainability_results[("C", model_name)] = run_task_explainability("C", model_name, run_bias_audit=True)


## Summary: faithfulness across tasks and architectures

Quick side-by-side of mean comprehensiveness/sufficiency for every
(task, model) that ran above — useful for the write-up's explainability
comparison table.

In [ ]:
summary_rows = []
for (task, model_name), result in explainability_results.items():
    if result is None:
        continue
    fdf = result["faithfulness"]
    summary_rows.append({
        "task": task,
        "model": MODEL_LABELS.get(model_name, model_name),
        "checkpoint": result["checkpoint_dir"],
        "mean_comprehensiveness": fdf["comprehensiveness"].mean(),
        "mean_sufficiency": fdf["sufficiency"].mean(),
    })

pd.DataFrame(summary_rows).sort_values(["task", "model"])
